## Momentum Persistance Test

### Research Ques: Do winners continue to outperform?

In [ ]:
import qnt.data as qndata
import numpy as np
import xarray as xr

data = qndata.cryptodaily_load_data(
    min_date="2015-01-01"
)

close = data.sel(field="close")
is_liquid = data.sel(field="is_liquid")

close = close.where(is_liquid == 1)

mom21 = close / close.shift(time=21) - 1

future21 = close.shift(time=-21) / close - 1

corr = xr.corr(
    mom21,
    future21,
    dim="time"
)

print("Momentum Persistence")
print(float(corr.mean()))

Momentum Persistence
0.021193217811877513

## Mean Reversion Test

### Research Ques: Do short-term losers rebound

In [ ]:
mom5 = close / close.shift(time=5) - 1

future5 = close.shift(time=-5) / close - 1

corr = xr.corr(
    mom5,
    future5,
    dim="time"
)

print("Mean Reversion Test")
print(float(corr.mean()))

Mean Reversion Test
-0.023646254661119682

## Regime Impact

### Research Ques: Do bull and bear markets behave differently?

In [ ]:
btc = close.sel(asset="BTC")

btc_sma200 = btc.rolling(time=200).mean()

bull_market = btc > btc_sma200

ret = close / close.shift(time=1) - 1

ret = ret.where(is_liquid == 1)

# remove inf values
ret = ret.where(np.isfinite(ret))

bull_ret = ret.where(bull_market)

bear_ret = ret.where(~bull_market)

print("Bull Market Average Return")
print(float(bull_ret.mean()))

print()

print("Bear Market Average Return")
print(float(bear_ret.mean()))

print()

print("Bull Market Volatility")
print(float(bull_ret.std()))

print()

print("Bear Market Volatility")
print(float(bear_ret.std()))

Bull Market Average Return
0.005030391062593629

Bear Market Average Return
-0.0019902694458312905

Bull Market Volatility
0.06421155819317155

Bear Market Volatility
0.053624366190387275

## Volume Leadership
### Research Ques: Do volume spikes predict future returns?

In [ ]:
vol = data.sel(field="vol")

vol_ma20 = vol.rolling(
    time=20
).mean()

vol_ratio = vol / vol_ma20

future5 = close.shift(time=-5) / close - 1

corr = xr.corr(
    vol_ratio,
    future5,
    dim="time"
)

print("Volume Leadership")
print(float(corr.mean()))

Volume Leadership
0.04808020533358951

## Volatility Compression
### Research Ques: Does low volatility precede large moves?

ret = close / close.shift(time=1) - 1

vol20 = ret.rolling(
    time=20
).std()

vol60 = ret.rolling(
    time=60
).std()

compression = vol20 / vol60

future_mag = abs(
    close.shift(time=-10) / close - 1
)

corr = xr.corr(
    compression,
    future_mag,
    dim="time"
)

print("Volatility Compression")
print(float(corr.mean()))

Volatility Compression
0.024185209702326634